<a href="https://colab.research.google.com/github/elninooooo/Agent/blob/main/vision_agents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Vision Agents with smolagents


This notebook is part of the [Hugging Face Agents Course](https://www.hf.co/learn/agents-course), a free Course from beginner to expert, where you learn to build Agents.

![Agents course share](https://huggingface.co/datasets/agents-course/course-images/resolve/main/en/communication/share.png)

## Let's install the dependencies and login to our HF account to access the Inference API

If you haven't installed `smolagents` yet, you can do so by running the following command:

In [1]:
!pip install smolagents

Let's also login to the Hugging Face Hub to have access to the Inference API.

In [2]:
from huggingface_hub import notebook_login

notebook_login()

## Providing Images at the Start of the Agent's Execution

In this approach, images are passed to the agent at the start and stored as `task_images` alongside the task prompt. The agent then processes these images throughout its execution.  

Consider the case where Alfred wants to verify the identities of the superheroes attending the party. He already has a dataset of images from previous parties with the names of the guests. Given a new visitor's image, the agent can compare it with the existing dataset and make a decision about letting them in.  

In this case, a guest is trying to enter, and Alfred suspects that this visitor might be The Joker impersonating Wonder Woman. Alfred needs to verify their identity to prevent anyone unwanted from entering.  

Let’s build the example. First, the images are loaded. In this case, we use images from Wikipedia to keep the example minimal, but imagine the possible use-cases!

In [7]:
from PIL import Image
from io import BytesIO
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

session = requests.Session()

retry = Retry(
    total=3,
    backoff_factor=5,
    status_forcelist=[429, 500, 502, 503, 504],
    respect_retry_after_header=True
)

session.mount("https://", HTTPAdapter(max_retries=retry))

images = []

for url in image_urls:
    response = session.get(
        url,
        headers=headers,
        timeout=30
    )

    response.raise_for_status()

    image = Image.open(
        BytesIO(response.content)
    ).convert("RGB")

    images.append(image)

Now that we have the images, the agent will tell us wether the guests is actually a superhero (Wonder Woman) or a villian (The Joker).

In [8]:
from google.colab import userdata
import os
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

In [17]:
from smolagents import CodeAgent, OpenAIServerModel

from smolagents import CodeAgent, OpenAIServerModel
from google.colab import userdata

model = OpenAIServerModel(
    model_id="gpt-4o",
    api_key=userdata.get("OPENAI_API_KEY").strip(),
    api_base="https://api.openai.com/v1"
)

# Instantiate the agent
agent = CodeAgent(
    tools=[],
    model=model,
    max_steps=20,
    verbosity_level=2
)

response = agent.run(
    """
    Describe the costume and makeup in these photos.
    Tell me if the guest is The Joker or Wonder Woman.

    You MUST return your response using the
    final_answer() function.

    Follow this format:

    <code>
    final_answer("Your description here")
    </code>
    """,
    images=images
)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Describe the costume and makeup in these photos.                                                                │
│     Tell me if the guest is The Joker or Wonder Woman.                                                          │
│                                                                                                                 │
│     You MUST return your response using the                                                                     │
│     final_answer() function.                                                                                    │
│                                                                                                                 │
│     Follow this format:                                                                                         │
│                                                                                                                 │
│     <code>                                                                                                      │
│     final_answer("Your description here")                                                                       │
│     </code>                                                                                                     │
│                                                                                                                 │
╰─ OpenAIModel - gpt-4o ──────────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
<code>                                                                                                             
final_answer("The costume and makeup in the photos are consistent with The Joker. The individual is wearing white  
face makeup with exaggerated red lips and green hair. The clothing includes a purple suit, highlighted by a bright 
shirt and tie, typical of The Joker's iconic appearance.")                                                         
                                                                                                                   

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("The costume and makeup in the photos are consistent with The Joker. The individual is wearing      
  white face makeup with exaggerated red lips and green hair. The clothing includes a purple suit, highlighted by  
  a bright shirt and tie, typical of The Joker's iconic appearance.")                                              
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: The costume and makeup in the photos are consistent with The Joker. The individual is wearing white 
face makeup with exaggerated red lips and green hair. The clothing includes a purple suit, highlighted by a bright 
shirt and tie, typical of The Joker's iconic appearance.

[Step 1: Duration 2.31 seconds| Input tokens: 3,049 | Output tokens: 60]

In [18]:
response

"The costume and makeup in the photos are consistent with The Joker. The individual is wearing white face makeup with exaggerated red lips and green hair. The clothing includes a purple suit, highlighted by a bright shirt and tie, typical of The Joker's iconic appearance."

In this case, the output reveals that the person is impersonating someone else, so we can prevent The Joker from entering the party!

## Providing Images with Dynamic Retrieval

This examples is provided as a `.py` file since it needs to be run locally since it'll browse the web. Go to the [Hugging Face Agents Course](https://www.hf.co/learn/agents-course) for more details.